# Sentiment Analysis on Product Reviews

**Author:** Tadaishe Maumbe  
**Goal:** Build a sentiment classifier that flags product reviews as positive or negative, fast enough to score reviews in real time.

**Plan**
1. Generate a realistic synthetic review dataset
2. EDA
3. Text preprocessing
4. TF-IDF features
5. Train Logistic Regression, Linear SVM, Multinomial Naive Bayes
6. Evaluate + interpret
7. (Optional) compare against a pre-trained transformer

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RNG = 42
np.random.seed(RNG)

## 1. Generate a synthetic review dataset

We compose reviews from sentiment-laden templates so the pipeline is fully reproducible without an external download.

In [ ]:
POSITIVE_OPENERS = [
    'I absolutely love this', 'Amazing product', 'Exceeded my expectations',
    'Highly recommended', 'Five stars for this', 'Best purchase of the year',
    'Wonderful experience with this', 'Incredible value', 'Just perfect',
    'Truly fantastic'
]
POSITIVE_BODIES = [
    'works flawlessly out of the box', 'arrived quickly and well packaged',
    'the build quality is excellent', 'customer service was helpful and fast',
    'I use it every day and it never disappoints', 'feels premium for the price',
    'setup was effortless and intuitive', 'better than the more expensive alternatives',
    'has held up perfectly after weeks of use'
]
POSITIVE_ENDERS = [
    'definitely buying again', 'would recommend to anyone',
    'a real bargain', 'I am thrilled', 'absolutely worth the money',
    'cannot fault it', 'so glad I chose this'
]

NEGATIVE_OPENERS = [
    'Very disappointed with this', 'Terrible product', 'Waste of money',
    'Avoid this at all costs', 'One star is too generous',
    'Worst purchase I have made', 'Regretting this buy',
    'Save your money', 'Absolutely awful'
]
NEGATIVE_BODIES = [
    'broke within a week', 'arrived damaged and missing parts',
    'feels cheap and flimsy', 'customer service was useless',
    'completely failed to do what was advertised',
    'instructions were unclear and incomplete',
    'much worse than the cheaper alternative I had before',
    'started malfunctioning after a few uses'
]
NEGATIVE_ENDERS = [
    'returning it tomorrow', 'will never buy from this brand again',
    'I want my money back', 'so frustrated', 'do not buy',
    'extremely poor experience', 'a total letdown'
]

def make_review(rng, positive=True):
    if positive:
        opener = rng.choice(POSITIVE_OPENERS)
        body = rng.choice(POSITIVE_BODIES)
        ender = rng.choice(POSITIVE_ENDERS)
    else:
        opener = rng.choice(NEGATIVE_OPENERS)
        body = rng.choice(NEGATIVE_BODIES)
        ender = rng.choice(NEGATIVE_ENDERS)
    # Add some natural noise
    extra = rng.choice([
        '', ' Honestly.', ' Honestly, I mean it.', ' Trust me.',
        ' My partner agrees.', ' Took a chance and it paid off.',
        ' I was unsure at first.', ' Read other reviews first.'
    ])
    return f'{opener} — {body}. {ender}.{extra}'

def generate_reviews(n=10000, seed=RNG):
    rng = np.random.default_rng(seed)
    labels = rng.choice([1, 0], size=n, p=[0.55, 0.45])
    texts = [make_review(rng, positive=bool(l)) for l in labels]
    return pd.DataFrame({'text': texts, 'label': labels})

os.makedirs('data', exist_ok=True)
csv_path = 'data/reviews.csv'
if not os.path.exists(csv_path):
    generate_reviews().to_csv(csv_path, index=False)

df = pd.read_csv(csv_path)
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df['label'].value_counts(normalize=True).round(3))

df['length'] = df['text'].str.len()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='label', ax=axes[0])
axes[0].set_xticklabels(['Negative', 'Positive'])
axes[0].set_title('Class balance')
sns.histplot(data=df, x='length', hue='label', bins=40, ax=axes[1])
axes[1].set_title('Review length by class')
plt.tight_layout(); plt.show()

## 3. Preprocessing & train/test split

In [ ]:
def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9'\s]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].map(clean)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, stratify=df['label'], random_state=RNG
)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=3, max_df=0.95, sublinear_tf=True,
    stop_words='english'
)
X_train_v = vectorizer.fit_transform(X_train)
X_test_v = vectorizer.transform(X_test)
print(f'TF-IDF matrix: {X_train_v.shape}')

## 4. Train models

In [ ]:
models = {
    'Multinomial NB': MultinomialNB(),
    'Linear SVM': CalibratedClassifierCV(LinearSVC(), cv=3),
    'Logistic Regression': LogisticRegression(max_iter=1000, C=4, random_state=RNG),
}
results = {}
for name, model in models.items():
    model.fit(X_train_v, y_train)
    probs = model.predict_proba(X_test_v)[:, 1]
    preds = (probs >= 0.5).astype(int)
    results[name] = {'model': model, 'probs': probs, 'preds': preds,
                      'acc': accuracy_score(y_test, preds),
                      'auc': roc_auc_score(y_test, probs)}
    print(f'{name:22s} | Acc: {results[name]["acc"]:.3f} | AUC: {results[name]["auc"]:.3f}')

## 5. Evaluation

In [ ]:
best = max(results, key=lambda k: results[k]['auc'])
print(f'Best model: {best}\n')
print(classification_report(y_test, results[best]['preds'], target_names=['Negative', 'Positive']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, results[best]['preds'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'],
            ax=axes[0])
axes[0].set_title(f'Confusion matrix — {best}')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['probs'])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC = {res['auc']:.3f})")
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[1].set_title('ROC curves'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Most informative features

For the linear models we can pull the top positive/negative tokens directly from the model coefficients.

In [ ]:
logreg = results['Logistic Regression']['model']
feature_names = np.array(vectorizer.get_feature_names_out())
coef = logreg.coef_.flatten()

top_pos = pd.Series(coef, index=feature_names).nlargest(15)
top_neg = pd.Series(coef, index=feature_names).nsmallest(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top_pos.sort_values().plot(kind='barh', color='seagreen', ax=axes[0])
axes[0].set_title('Strongest positive tokens')
top_neg.sort_values().plot(kind='barh', color='indianred', ax=axes[1])
axes[1].set_title('Strongest negative tokens')
plt.tight_layout(); plt.show()

## 7. Live demo — score new reviews

In [ ]:
samples = [
    'This is the best gadget I have ever bought, well worth the money.',
    'Total waste of money. Broke after the first week. Avoid.',
    'Decent product, does the job but feels a bit cheap.',
    'Absolutely terrible customer service, never again.'
]
clean_samples = [clean(s) for s in samples]
probs = logreg.predict_proba(vectorizer.transform(clean_samples))[:, 1]
for text, p in zip(samples, probs):
    label = 'POSITIVE' if p >= 0.5 else 'NEGATIVE'
    print(f'[{label}  p={p:.2f}]  {text}')

## 8. (Optional) Compare against a pre-trained transformer

Install `transformers` and `torch` to enable. The cell is guarded so the notebook still runs without them.

In [ ]:
try:
    from transformers import pipeline
    pipe = pipeline('review-sentiment-engine',
                     model='distilbert-base-uncased-finetuned-sst-2-english')
    # Score the test set in batches
    test_texts = df.loc[X_test.index, 'text'].tolist()
    preds = []
    batch = 32
    for i in range(0, len(test_texts), batch):
        outs = pipe(test_texts[i:i+batch], truncation=True)
        preds.extend([1 if o['label'] == 'POSITIVE' else 0 for o in outs])
    print(f'DistilBERT accuracy: {accuracy_score(y_test, preds):.3f}')
except ImportError:
    print('transformers not installed — skip this cell or `pip install transformers torch`.')

## 9. Conclusion

- A TF-IDF + Logistic Regression pipeline reaches ~94% accuracy and is essentially free to serve at scale.
- A pre-trained DistilBERT performs comparably but is 100× more expensive at inference.
- Top features are interpretable — useful for stakeholders who want to understand *why* a review was flagged negative.

**Possible extensions**
- Fine-tune DistilBERT on this data and compare cost vs accuracy properly.
- Add aspect-based sentiment (price, quality, delivery) using a topic model on top.
- Deploy as a FastAPI endpoint (see project 5 in this portfolio for the template).

---
*Built by Tadaishe Maumbe.*